# Virtual stores with the `vds` plugin

This notebook replaces the old Icechunk tutorial: build a virtual store from NASA
granules, read it back, and append new granules — **without copying any data**.

The plugin replaces the **write** half of the old workflow
(`virtualize()` + `write_virtual()`). Reading stays on the specialized
`open_virtual()` — with `ea.open(asset)` as shorthand for the common case:

```python
job = ea.plugins.vds.request(granules, store="mur_sst.icechunk", concat_dim="time")
asset = job.results()[0]              # the store, as an Asset
ds = ea.open_virtual(asset.href)      # read it back (or ea.open(asset))
```

The request returns a `Job` immediately, so you can watch it work
(`job.info()`), open the store when it is done, and call the same request again
with a new batch to **append** — the store's existence decides create vs append.
Pass an existing `distributed.Client` (`client=client`) to run the per-granule
opens on a Dask cluster; this notebook creates a local one.

Install the plugin first (it is not on PyPI yet):

```bash
pip install "earthaccess-plugin-vds[distributed] @ git+https://github.com/betolink/earthaccess-plugin-vds"
```

The `distributed` extra also installs `dask-labextension`, so the Dask
cluster and its dashboard appear in the JupyterLab launcher and sidebar
(restart JupyterLab after installing).


In [1]:
import earthaccess

earthaccess.login()

Auth(authenticated=True, user='earthaccess', strategy='netrc', system=urs.earthdata.nasa.gov)

## 1. Build a virtual dataset from granules

MUR SST is a level-4, globally gridded SST product — a good fit for
virtualizing: the granules are homogeneous, CF-compliant, and share a unified
grid.

In [2]:
sst_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-02", "2013-01-31")
)
len(sst_granules)

20

## 2. Start a Dask cluster (generic Dask)

The request accepts an existing `distributed.Client`: per-granule opens are
submitted to it, and `job.info()["dashboard"]` points at its dashboard. This
notebook uses a local cluster; the same code works with any Dask deployment.

Workers read the data themselves, so **credentials must exist where they run**:
a `~/.netrc` entry (this laptop) or `EARTHDATA_TOKEN` / `EARTHDATA_USERNAME` +
`EARTHDATA_PASSWORD`. For a remote cluster, send them explicitly with
`ea.distribute_credentials({...}, cluster=cluster)` (Coiled uses its private
channel).

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, processes=True)
client = Client(cluster)
client

## 3. Create the store and watch the request

`request()` returns a `Job` right away. While it runs, `job.info()` reports
which phase it is in — `virtualize` (opening granules), `combine`, or `write` —
plus how many granules have been opened, the total, and the percent complete.
`show_progress=True` also draws a per-granule progress bar.

The result is an `Asset` whose `href` is the store (`format="icechunk"`,
`access="file"` for a local directory). Nothing was downloaded: the store holds
byte-range references to the original NASA files.

In [3]:
import time

TERMINAL = {"SUCCEEDED", "PARTIAL", "FAILED", "CANCELED", "PAUSED"}


def wait_with_progress(job, interval=2.0):
    """Poll a job and print its phase + per-granule counts while it runs."""
    while job.status().name not in TERMINAL:
        info = job.info()
        print(
            f"{info['phase']:>10}  {info['opened']}/{info['total']} granules"
            f"  {info['percent']:5.1f}%"
        )
        time.sleep(interval)
    info = job.info()
    print(
        f"{job.status().name}  {info['opened']}/{info['total']} granules"
        f"  {info['percent']:5.1f}%"
    )
    if info.get("error"):
        print(info["error"])
    return job

In [4]:
job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst.icechunk",
    concat_dim="time",
    show_progress=True,
    client=client,
)
wait_with_progress(job)

virtualize  0/31 granules    0.0%


/home/betolink/github/betolink/earthaccess/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
virtualize:   0%|                                                                                                                                                                                                                  | 0/31 [00:00<?, ?granule/s]

virtualize  0/31 granules    0.0%
virtualize  0/31 granules    0.0%
virtualize  0/31 granules    0.0%
virtualize  0/31 granules    0.0%
virtualize  0/31 granules    0.0%
virtualize  0/31 granules    0.0%


virtualize:  45%|██████████████████████████████████████████████████████████████████████████████████████████▊                                                                                                              | 14/31 [00:13<00:08,  2.05granule/s]

virtualize  19/31 granules   61.3%


virtualize:  65%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                                       | 20/31 [00:14<00:03,  2.81granule/s]

virtualize  20/31 granules   64.5%
virtualize  20/31 granules   64.5%
virtualize  20/31 granules   64.5%
virtualize  20/31 granules   64.5%
virtualize  20/31 granules   64.5%


virtualize: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31/31 [00:25<00:00,  1.80granule/s]  2026-11-02T21:29:56.169457Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-02T21:29:56.236460Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:31

virtualize: 100%|████████████████████████

SUCCEEDED  31/31 granules  100.0%


<VirtualJob vds:1f4c10cfafe44f3e99e2e0f9d1ff11d7 SUCCEEDED>

In [5]:
asset = job.results()[0]
asset

Asset(href='mur_sst.icechunk', access='file', auth='none', format='icechunk')

## 4. Read the store back

`open_virtual()` recognizes an icechunk store, authorizes its virtual chunk
containers with your EDL credentials, and returns an xarray dataset. Reads are
served straight from the original NASA files. `ea.open(asset)` is the generic
shorthand when you do not need the specialized reader options
(`access=`, `force_external=`, `authorize_virtual_chunk_access=`,
`storage_options=`, `load=False`).

In [7]:
ds = earthaccess.open_virtual(asset.href)
print(f"Dataset spanning from {ds.time.min().values} to {ds.time.max().values}")
ds

Dataset spanning from 2013-01-02T09:00:00.000000000 to 2013-02-01T09:00:00.000000000


  2026-11-02T21:30:38.730623Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



<xarray.Dataset> Size: 562GB
Dimensions:           (time: 31, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 248B 2013-01-02T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysis_error    (time, lat, lon) float64 161GB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
    mask              (time, lat, lon) float32 80GB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
    sea_ice_fraction  (time, lat, lon) float64 161GB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
    analysed_sst      (time, lat, lon) float64 161GB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

`load=False` gives the **manifest form** instead: variables are
`ManifestArray`s pointing at byte ranges in the original files, with concrete
coordinates so you can combine and write them again. This is the same view the
appender uses.

In [ ]:
references = earthaccess.open_virtual(asset.href, load=False)
name = next(iter(references.data_vars))
print(name, "->", type(references[name].data).__name__)
print(f"{references.virtualize.nbytes / 1e6:.1f} MB referenced, 0 copied")
references

## 5. Append the next month

Search the following time step and send **the same request**, with
`store=asset.href`. Because the store already exists, the write is an append
along `concat_dim` (`append_dim` defaults to it) — no flag needed. The href
stays the same and icechunk records the update as a new commit on `main`.

In [ ]:
new_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-02-01", "2013-02-28")
)
len(new_granules)

In [ ]:
append_job = earthaccess.plugins.vds.request(
    new_granules,
    store=asset.href,
    concat_dim="time",
    show_progress=True,
    client=client,
)
wait_with_progress(append_job)

asset = append_job.results()[0]
asset

In [ ]:
updated_ds = earthaccess.open_virtual(asset.href)
print(
    f"Dataset now spanning from {updated_ds.time.min().values} to "
    f"{updated_ds.time.max().values}"
)

## Advanced: controlling the icechunk repository

`request()` creates the icechunk repository for you with
`icechunk.RepositoryConfig.default()`. Three optional arguments give you the
repository you want without leaving the plugin:

- `repository_config=` — the `RepositoryConfig` used **when the repository is
  created**: manifest splitting, preload, inline-chunk threshold, compression,
  concurrency limits. It is ignored (with a warning) when the store already
  exists, because icechunk keeps the stored configuration on append.
- `store=` an `icechunk.Repository` (plus `href=` for the returned `Asset`) —
  bring a repository you created yourself; the plugin writes into it and leaves
  its configuration alone.
- `virtual_chunk_containers=` — extra containers keyed by `url_prefix`, merged
  over the ones derived from the chunk references. Use it for private buckets,
  custom endpoints or non-default regions.

Reading is the other half: `ea.open_virtual()` authorizes the containers stored
in the repository, and `authorize_virtual_chunk_access=` overrides the
credentials it uses.

In [ ]:
import icechunk

# Split manifests per array/dimension and keep small chunks inline.
config = icechunk.RepositoryConfig.default()
config.inline_chunk_threshold_bytes = 1024
config.manifest = icechunk.ManifestConfig(
    splitting=icechunk.ManifestSplittingConfig.from_dict(
        {
            icechunk.ManifestSplitCondition.name_matches("analysed_sst"): {
                icechunk.ManifestSplitDimCondition.DimensionName("time"): 32,
            },
        }
    )
)

job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst_tuned.icechunk",
    concat_dim="time",
    repository_config=config,  # applied only when the repository is created
    client=client,
)
tuned = job.results()[0]
tuned

In [ ]:
# Bring your own repository: create it with the configuration you want, then
# hand it to the plugin. `href=` tells the Asset where it lives, because an
# icechunk.Repository carries no readable path.
storage = icechunk.Storage.new_local_filesystem("mur_sst_owned.icechunk")
repo = icechunk.Repository.create(storage, config=config)

job = earthaccess.plugins.vds.request(
    sst_granules,
    store=repo,  # an icechunk.Repository, not a path
    href="mur_sst_owned.icechunk",
    concat_dim="time",
    client=client,
)
owned = job.results()[0]
owned

In [ ]:
# Extra containers: private buckets, custom endpoints, other regions. Entries
# here win over the containers derived from the granules' chunk references.
job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst_private.icechunk",
    concat_dim="time",
    virtual_chunk_containers={
        "s3://my-private-bucket/": icechunk.s3_store(region="us-east-1"),
        "https://data.example.com/": icechunk.http_store(
            headers={"Authorization": "Bearer <token>"}
        ),
    },
    client=client,
)
private = job.results()[0]

In [ ]:
# Readers must authorize the containers the store references. earthaccess
# derives that mapping from the repository config; override it to change the
# credentials (keys match the container url_prefix exactly).
ds = earthaccess.open_virtual(
    tuned.href,
    authorize_virtual_chunk_access={
        "https://<container-host>/": icechunk.credentials.HttpAccess,
    },
)
ds

## 6. The same workflow with kerchunk-parquet

`format="kerchunk"` writes a references **directory** whose name ends in
`.parquet` (that is what `open_virtual()`/`ea.open()` sniff). Kerchunk has no in-place append:
appending reads the old references, combines them with the delta, and writes a
**new versioned directory** — `mur_sst_refs.parquet` → `mur_sst_refs_v2.parquet`
— so the old version is preserved. Pass `replace=True` to rewrite in place and
keep a stable href.

In [ ]:
kerchunk_job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst_refs.parquet",
    format="kerchunk",
    concat_dim="time",
    show_progress=True,
    client=client,
)
wait_with_progress(kerchunk_job)

kerchunk_asset = kerchunk_job.results()[0]
kerchunk_asset

In [ ]:
kerchunk_ds = earthaccess.open_virtual(kerchunk_asset.href)
print(
    f"Kerchunk store spanning from {kerchunk_ds.time.min().values} to "
    f"{kerchunk_ds.time.max().values}"
)

In [ ]:
kerchunk_append = earthaccess.plugins.vds.request(
    new_granules,
    store=kerchunk_asset.href,
    format="kerchunk",
    concat_dim="time",
    show_progress=True,
    client=client,
)
wait_with_progress(kerchunk_append)

kerchunk_asset = kerchunk_append.results()[0]
kerchunk_asset.href  # a new versioned directory, e.g. mur_sst_refs_v2.parquet

In [ ]:
kerchunk_updated = earthaccess.open_virtual(kerchunk_asset.href)
print(
    f"Kerchunk store now spanning from "
    f"{kerchunk_updated.time.min().values} to {kerchunk_updated.time.max().values}"
)

## Progress and cancellation

- `job.info()` always returns the same shape:
  `{"status", "opened", "total", "failed", "percent", "phase", "dashboard"}`.
- `phase` goes `virtualize → combine → write → done`.
- With `client=client`, each granule is opened on the cluster; the counters and
  the `tqdm` bar (`show_progress=True`) update as workers finish, and
  `info()["dashboard"]` links to the Dask dashboard.
- Workers resolve their own credentials (`~/.netrc`, `EARTHDATA_TOKEN`, or
  `EARTHDATA_USERNAME`/`EARTHDATA_PASSWORD`). For remote clusters, call
  `ea.distribute_credentials({...}, cluster=cluster)` first — it uses Coiled's
  private channel when available and never puts secrets in task payloads.
- `job.cancel()` cancels per-granule opens that have not started yet
  (`client.cancel` when a client is passed).


## Summary

| Old API | `vds` plugin |
|---|---|
| `ea.virtualize(granules, concat_dim="time")` | `ea.plugins.vds.request(granules, concat_dim="time").results()` |
| `ea.write_virtual(vds, "store")` | `ea.plugins.vds.request(granules, store="store.icechunk", concat_dim="time").results()` → `[Asset]` |
| `ea.write_virtual(vds, "store", append_dim="time")` | the same request against the existing store (`append_dim` defaults to `concat_dim`) |
| `ea.open_virtual("store")` | **unchanged** — still the specialized reader; `ea.open(asset)` is the shorthand |
| `format="kerchunk"` reference file | `format="kerchunk"` with a `*.parquet` directory; appends make a versioned sibling |
| `parallel="dask"` (implicit cluster) | create a `Client` and pass `client=client` to the request |
| `open_virtual(load=False)` | `open_virtual(asset.href, load=False)` — manifest form for **both** formats |

The store is versioned (icechunk): every update is a new commit on `main`, so
you can inspect history, roll back, or share the store with collaborators who
have access to the underlying data.


## Clean up

Close the client and the local cluster when you are done. Your stores on disk
and in the cloud are untouched.

In [ ]:
client.close()
cluster.close()